In [3]:
import pandas as pd

df = pd.read_csv("train.csv")
print(df.head()) # 只看前五行

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


Survived：0 = 遇难，1 = 生还。
Pclass：客舱等级（1 = 头等舱，2 = 二等舱，3 = 三等舱。这代表了乘客的社会地位）。
Name：姓名。
Sex：性别。
Age：年龄。
SibSp：该乘客在船上的 兄弟姐妹/配偶 数量。
Parch：该乘客在船上的 父母/孩子 数量。
Ticket：船票号码。
Fare：买船票花的钱。
Cabin：客舱编号。
Embarked：登船的港口（C = 瑟堡，Q = 皇后镇，S = 南安普顿）。

年龄当中的缺失值我们选择用中位数来填，因为平均数很容易受极端数据的影响，导致数据不准确

In [4]:
median_age = df["Age"].median()
df['Age'] = df["Age"].fillna(median_age)

男性和女性我们分别用0和1代替
在泰坦尼克号那个“女士优先”的背景下，女性的生还率比男性更高，所以当机器计算权重w时，算出来的w大概率是一个很大的整数

In [5]:
df['Sex'] = df['Sex'].map({'female': 1, 'male': 0})
print(df[['Name', 'Sex']].head())

                                                Name  Sex
0                            Braund, Mr. Owen Harris    0
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1
2                             Heikkinen, Miss. Laina    1
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1
4                           Allen, Mr. William Henry    0


对于上岸港口，对生还率没有任何影响，所以我们把原数据转换为列表，列表元素等于上岸港口的个数，对应的港口，体现在对应列表中的位置对应元素为1

In [6]:
df = pd.get_dummies(df, columns=['Embarked'])
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name  Sex   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris    0  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1  38.0      1      0   
2                             Heikkinen, Miss. Laina    1  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1  35.0      1      0   
4                           Allen, Mr. William Henry    0  35.0      0      0   

             Ticket     Fare Cabin  Embarked_C  Embarked_Q  Embarked_S  
0         A/5 21171   7.2500   NaN       False       False        True  
1          PC 17599  71.2833   C85        True       False       False  
2  STON/O2. 3101282   7.9250   NaN       False       False        True  
3   

最后，我们可以用drop()函数来清除掉与判断完全无关的数据（噪音）

In [7]:
df = df.drop(columns = ['PassengerId', 'Name','Ticket','Cabin'])
print(df.info())  # 输出表格里的所有字段以及对应变量类型

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Sex         891 non-null    int64  
 3   Age         891 non-null    float64
 4   SibSp       891 non-null    int64  
 5   Parch       891 non-null    int64  
 6   Fare        891 non-null    float64
 7   Embarked_C  891 non-null    bool   
 8   Embarked_Q  891 non-null    bool   
 9   Embarked_S  891 non-null    bool   
dtypes: bool(3), float64(2), int64(5)
memory usage: 51.5 KB
None


下一步我们要把特征和标签分开，单独拎出特征和标签

In [8]:
X = df.drop(columns=['Survived'])
y = df['Survived']

print(f"考题 X 的形状：{X.shape} (几百个乘客，每个乘客有几个特征)")
print(f"答案 y 的形状：{y.shape} (几百个 0 或 1 的结局)")

考题 X 的形状：(891, 9) (几百个乘客，每个乘客有几个特征)
答案 y 的形状：(891,) (几百个 0 或 1 的结局)


接下来就是常规的模型训练了

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"用来训练的 X_train: {X_train.shape[0]} 人")
print(f"留作验证的 X_val:   {X_val.shape[0]} 人")

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)
y_pred = model.predict(X_val)

acc = accuracy_score(y_val, y_pred)
print(f"🏆 本地验证集准确率 (Accuracy): {acc * 100:.2f}%")

用来训练的 X_train: 712 人
留作验证的 X_val:   179 人
🏆 本地验证集准确率 (Accuracy): 81.01%


下面我们来看一下哪些特征是对逃生有最大正作用的，哪些是对逃生有最大负作用的

In [10]:
import pandas as pd

feature_names = X.columns  # 提取所有的特征，赋值给feature_names，返回的是一个列表
weights = model.coef_[0]  # scikit-learn的权重列表是一个嵌套列表，所以只能用中括号的形式拉取列表

importance_df = pd.DataFrame({
    '特征':feature_names,
    '权重':weights
})

importance_df = importance_df.sort_values(by='权重', ascending=False)
print(importance_df)

           特征        权重
1         Sex  2.590228
6  Embarked_C  0.118032
5        Fare  0.002528
7  Embarked_Q -0.027426
2         Age -0.030508
4       Parch -0.107868
3       SibSp -0.293289
8  Embarked_S -0.309045
0      Pclass -0.935236
